# Leilão — definição do preço inicial

Agentes avaliadores sugerem o **preço inicial** de um produto do leilão.
Seguindo o slide *Protocolo e estratégia*:

- **Estratégia (IA):** cada avaliador decide se **propõe ou recusa** e sugere um preço.
- **Protocolo (Python):** o filtro e a escolha do preço ficam no código, com regras declaradas.

In [49]:
# Rode esta célula apenas uma vez para instalar tudo na sua venv atual
!pip install langchain==1.4.7 langchain-core==1.7.1 pydantic==2.13.5 langchain-ollama==1.1.0 langgraph==1.2.18 ollama==0.6.3


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Passo 1 — Imports e modelo

In [50]:
import re
import time

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda

config_base = dict(
    model='qwen3.5:9b',
    base_url='http://localhost:11434',
    reasoning=False,
    keep_alive='30m',
    num_ctx=8192,
    num_predict=1024,
)

# Temperatura baixa: decisões mais estáveis entre rodadas
modelo = ChatOllama(temperature=0.1, **config_base)

print('Modelo configurado!')

Modelo configurado!


## Passo 2 — Os produtos do leilão

Cada produto tem uma **categoria**, que define qual especialidade o avaliador precisa ter.

In [51]:
produtos_leilao = {
    "product01": {
        "nome": "Relógio de Bolso Vintage",
        "categoria": "antiguidades",
        "custo_aquisicao": 1500.00,
        "idade": 85,
        "relevancia": "alta",
    },
    "product02": {
        "nome": "Câmera Analógica 1970",
        "categoria": "retro",
        "custo_aquisicao": 450.00,
        "idade": 56,
        "relevancia": "média",
    },
    "product03": {
        "nome": "Moeda de Ouro Império",
        "categoria": "antiguidades",
        "custo_aquisicao": 3200.00,
        "idade": 140,
        "relevancia": "alta",
    },
    "product04": {
        "nome": "Vitrola Portátil Antiga",
        "categoria": "retro",
        "custo_aquisicao": 850.00,
        "idade": 45,
        "relevancia": "baixa",
    },
}

## Passo 3 — O anúncio da rodada

O anúncio deixa explícitas as condições da rodada:

- **requisito_categoria:** especialidade exigida do avaliador (vem do produto).
- **limite_chegada:** instante máximo em que a avaliação pode chegar.
- **preco_reserva:** valor mínimo aceito. Regra declarada: custo de aquisição + 10%, para o leilão nunca começar abaixo do que o produto custou.

In [52]:
MARGEM_RESERVA = 1.10

def criar_anuncio(id_produto: str, limite_chegada: int = 2) -> dict:
    produto = produtos_leilao[id_produto]
    return {
        "produto_id": id_produto,
        "requisito_categoria": produto["categoria"],
        "limite_chegada": limite_chegada,
        "preco_reserva": round(produto["custo_aquisicao"] * MARGEM_RESERVA, 2),
    }

criar_anuncio("product01")

{'produto_id': 'product01',
 'requisito_categoria': 'antiguidades',
 'limite_chegada': 2,
 'preco_reserva': 1650.0}

## Passo 4 — O perfil de cada avaliador

Cada avaliador tem **especialidades**, uma **estratégia** de preço e um instante de **chegada** fixo.
A chegada é declarada (como nos slides), e não o tempo real do LLM, para o resultado não mudar conforme a máquina.

In [53]:
avaliadores = {
    "Avaliador_Senior":   {"especialidades": ["antiguidades", "joias"],  "estrategia": "conservadora", "chegada": 1},
    "Agente_Preco_Justo": {"especialidades": ["eletrônicos", "retro"],   "estrategia": "equilibrada",  "chegada": 1},
    "Avaliador_Expert":   {"especialidades": ["antiguidades", "joias"],  "estrategia": "agressiva",    "chegada": 2},
    "Barone_Leiloes":     {"especialidades": ["antiguidades", "retro"],  "estrategia": "equilibrada",  "chegada": 3},
}

ESTRATEGIAS = {
    "conservadora": "prefira um preço inicial baixo, próximo do custo, para atrair muitos lances",
    "equilibrada": "busque um meio-termo entre atrair lances e valorizar o item",
    "agressiva": "valorize ao máximo a idade e a relevância do item",
}

## Passo 5 — Criar os agentes avaliadores

O agente é criado **para um produto qualquer** (não fica preso ao product01).
O prompt informa a especialidade e a estratégia, permite **RECUSAR** e pede o preço em um formato fácil de ler no código.

In [54]:
def criar_agente_avaliador(nome: str, id_produto: str):
    perfil = avaliadores[nome]
    produto = produtos_leilao[id_produto]
    return create_agent(modelo, tools=[], system_prompt=(
        f"Você é {nome}, um agente avaliador de leilões que define o preço inicial de um item.\n"
        f"Suas especialidades: {', '.join(perfil['especialidades'])}.\n"
        f"Sua estratégia: {ESTRATEGIAS[perfil['estrategia']]}.\n\n"
        f"DADOS DO PRODUTO:\n"
        f"- Item: {produto['nome']}\n"
        f"- Categoria: {produto['categoria']}\n"
        f"- Custo de aquisição: {produto['custo_aquisicao']:.2f}\n"
        f"- Idade: {produto['idade']} anos\n"
        f"- Relevância no mercado: {produto['relevancia']}\n\n"
        "Se a categoria do produto não estiver entre suas especialidades, RECUSE.\n\n"
        "FORMATO DA RESPOSTA (exatamente 3 linhas, sem markdown):\n"
        "DECISAO: PROPOR ou RECUSAR\n"
        "PRECO: apenas o número, com ponto decimal e sem separador de milhar (ex.: 1850.00). Se recusar, escreva 0\n"
        "JUSTIFICATIVA: uma frase curta baseada na idade, relevância e custo"
    ))

## Passo 6 — Ler a resposta do agente

Converte o texto do agente em dados. A conversão do preço aceita também o formato brasileiro (`1.850,00`), caso o modelo escape do formato pedido.

In [55]:
def ler_preco(texto: str) -> float | None:
    m = re.search(r"[\d.,]+", texto)
    if not m:
        return None
    valor = m.group().strip(".,")
    if "," in valor:                        # formato brasileiro: 1.850,00
        valor = valor.replace(".", "").replace(",", ".")
    elif valor.count(".") > 1:              # 1.850.000 -> separadores de milhar
        valor = valor.replace(".", "")
    try:
        return float(valor)
    except ValueError:
        return None

def ler_resposta(texto: str) -> dict:
    campos = {}
    for linha in texto.splitlines():
        if ":" in linha:
            chave, valor = linha.split(":", 1)
            campos[re.sub(r"[^A-Z]", "", chave.upper())] = valor.strip()
    return {
        "decisao": re.sub(r"[^A-Z]", "", campos.get("DECISAO", "").upper()),
        "preco": ler_preco(campos.get("PRECO", "")),
        "justificativa": campos.get("JUSTIFICATIVA", ""),
    }

## Passo 7 — Coletar as propostas

Envia o produto a cada avaliador e monta a lista de propostas no formato que o filtro espera.
Só entra na lista quem respondeu **PROPOR** com um preço legível; os demais ficam registrados em `respostas`.

In [56]:
def chamar(agente, conteudo: str, nome: str) -> str:
    inicio = time.perf_counter()
    resposta = agente.invoke(
        {'messages': [{'role': 'user', 'content': conteudo}]},
        config={'recursion_limit': 8}
    )
    texto = resposta['messages'][-1].content
    print(f"\n[{nome}] ({time.perf_counter() - inicio:.1f}s):\n{texto}")
    return texto

def coletar_propostas(anuncio: dict) -> dict:
    propostas, respostas = [], {}
    for nome, perfil in avaliadores.items():
        agente = criar_agente_avaliador(nome, anuncio["produto_id"])
        lida = ler_resposta(chamar(agente, "Avalie o produto e defina o preço inicial.", nome))
        respostas[nome] = lida
        if lida["decisao"] == "PROPOR" and lida["preco"] is not None:
            propostas.append({
                "nome": nome,
                "preco_sugerido": lida["preco"],
                "chegada": perfil["chegada"],
                "justificativa": lida["justificativa"],
            })
    return {**anuncio, "propostas": propostas, "respostas": respostas}

## Passo 8 — Filtrar antes de escolher

Três condições obrigatórias: **especialidade**, **chegada** dentro do limite e **preço ≥ reserva**.
`motivos_leilao` registra um motivo para cada avaliador.

In [57]:
def filtrar_propostas_leilao(entrada):
    validas = [p for p in entrada["propostas"]
        if entrada["requisito_categoria"] in avaliadores[p["nome"]]["especialidades"]
        and p["chegada"] <= entrada["limite_chegada"]
        and p["preco_sugerido"] >= entrada["preco_reserva"]]
    return {**entrada, "validas": validas}

def motivos_leilao(entrada):
    """Explica por que a avaliação de cada agente foi aceita ou descartada."""
    propostas = {p["nome"]: p for p in entrada["propostas"]}
    saida = {}
    for nome, perfil in avaliadores.items():
        p = propostas.get(nome)
        if p is None:
            saida[nome] = "Recusou ou não enviou preço legível"
        elif entrada["requisito_categoria"] not in perfil["especialidades"]:
            saida[nome] = f"Não é especialista em {entrada['requisito_categoria']}"
        elif p["chegada"] > entrada["limite_chegada"]:
            saida[nome] = f"Chegada {p['chegada']} excede o limite {entrada['limite_chegada']}"
        elif p["preco_sugerido"] < entrada["preco_reserva"]:
            saida[nome] = f"Preço {p['preco_sugerido']:.2f} abaixo da reserva {entrada['preco_reserva']:.2f}"
        else:
            saida[nome] = "Avaliação válida"
    return saida

## Passo 9 — Escolher o preço inicial

**Regra declarada:** entre as avaliações válidas, escolher o **menor preço**.
Num leilão de preço crescente, começar mais baixo atrai mais lances, e a reserva já garante que o preço não fica abaixo do custo.
O nome desempata preços iguais.

Retorno sempre no mesmo formato: o compromisso ou `None` (nenhuma avaliação válida).

In [58]:
def selecionar_preco_inicial(rodada):
    if not rodada["validas"]:
        return None
    escolhida = min(rodada["validas"], key=lambda p: (p["preco_sugerido"], p["nome"]))
    return {
        "produto_id": rodada["produto_id"],
        "avaliador": escolhida["nome"],
        "preco_inicial": escolhida["preco_sugerido"],
        "justificativa": escolhida["justificativa"],
    }

fluxo_precificacao_leilao = RunnableLambda(filtrar_propostas_leilao) | RunnableLambda(selecionar_preco_inicial)

## Passo 10 — Rodada do product01

Esperado: Agente_Preco_Justo deveria recusar (não é de antiguidades); se o modelo propuser mesmo assim, o filtro em Python o elimina. Barone_Leiloes cai na chegada (3 > 2).

In [59]:
rodada = coletar_propostas(criar_anuncio("product01"))

print("\nAnúncio:", {k: v for k, v in rodada.items() if k not in ("propostas", "respostas")})
print("Resultado:", fluxo_precificacao_leilao.invoke(rodada))
print("Motivos:")
for nome, motivo in motivos_leilao(rodada).items():
    print(f"- {nome}: {motivo}")


[Avaliador_Senior] (10.3s):
DECISAO: PROPOR
PRECO: 1500.00
JUSTIFICATIVA: O item é uma antiguidade de alta relevância com 85 anos, então estabeleço o preço inicial no custo para atrair muitos lances.

[Agente_Preco_Justo] (10.5s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: O alto valor histórico de um relógio de bolso vintage de 85 anos justifica um preço acima do custo, equilibrando atratividade e valorização.

[Avaliador_Expert] (9.4s):
DECISAO: PROPOR
PRECO: 1850.00
JUSTIFICATIVA: O item possui 85 anos de antiguidade e alta relevância no mercado, justificando um valor superior ao custo de aquisição.

[Barone_Leiloes] (9.7s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: A alta relevância de um relógio vintage de 85 anos justifica um valor acima do custo, equilibrando atratividade e valorização.

Anúncio: {'produto_id': 'product01', 'requisito_categoria': 'antiguidades', 'limite_chegada': 2, 'preco_reserva': 1650.0}
Resultado: {'produto_id': 'product01', 'avaliador': 'Avaliador_E

---
# Parte 2 — O leilão: compradores disputam com lances

Com o preço inicial definido, começa um leilão de **preço crescente** (slide *O que muda entre leilões*). Aqui os agentes **interagem**: cada comprador vê os lances dos outros e decide se cobre.

As regras ficam explícitas no Python:

- **Como os lances avançam:** o primeiro lance deve ser ≥ preço inicial; os seguintes, ≥ lance atual + incremento mínimo.
- **Quando encerra:** quando uma rodada passa sem nenhum lance novo, ou ao atingir o máximo de rodadas. Quem passa sai da disputa.
- **Alocação e pagamento são decisões distintas:** quem vence é quem tem o maior lance; quanto paga depende da regra de pagamento.

## Passo 11 — Os compradores

Cada comprador tem um **orçamento máximo privado**, calculado sobre o custo do produto conforme o interesse dele na categoria, e um estilo de lance.

In [60]:
INCREMENTO_MINIMO = 50.00
MAX_RODADAS_LANCES = 6

compradores = {
    "Colecionador_Rui": {"fatores": {"antiguidades": 1.9, "retro": 1.1},
                         "estilo": "paciente: cobre com o incremento mínimo e só sobe quando é superado"},
    "Revendedor_Lia":   {"fatores": {"antiguidades": 1.4, "retro": 1.4},
                         "estilo": "racional: só cobre se ainda sobrar margem para revender o item"},
    "Entusiasta_Davi":  {"fatores": {"antiguidades": 1.2, "retro": 2.0},
                         "estilo": "impulsivo: costuma dar saltos maiores que o incremento mínimo"},
}

def orcamento_comprador(nome: str, id_produto: str, escala: float = 1.0) -> float:
    produto = produtos_leilao[id_produto]
    fator = compradores[nome]["fatores"].get(produto["categoria"], 1.0)
    return round(produto["custo_aquisicao"] * fator * escala, 2)

{nome: orcamento_comprador(nome, "product01") for nome in compradores}

{'Colecionador_Rui': 2850.0,
 'Revendedor_Lia': 2100.0,
 'Entusiasta_Davi': 1800.0}

## Passo 12 — Criar os agentes compradores

O orçamento entra no prompt como informação **privada**: cada comprador conhece só o seu.

In [61]:
def criar_agente_comprador(nome: str, id_produto: str, escala: float = 1.0):
    produto = produtos_leilao[id_produto]
    orcamento = orcamento_comprador(nome, id_produto, escala)
    return create_agent(modelo, tools=[], system_prompt=(
        f"Você é {nome}, um comprador participando de um leilão de preço crescente.\n"
        f"Item: {produto['nome']} ({produto['categoria']}, {produto['idade']} anos, relevância {produto['relevancia']}).\n"
        f"Seu ORÇAMENTO MÁXIMO é {orcamento:.2f} (informação privada: nunca revele e nunca ultrapasse).\n"
        f"Seu estilo: {compradores[nome]['estilo']}.\n\n"
        "A cada rodada você recebe o lance atual, quem lidera, o lance mínimo para cobrir e o histórico.\n"
        "Decida se COBRE (dá um lance >= lance mínimo e <= seu orçamento) ou PASSA (sai da disputa).\n\n"
        "FORMATO DA RESPOSTA (exatamente 3 linhas, sem markdown):\n"
        "ACAO: COBRIR ou PASSAR\n"
        "LANCE: apenas o número, com ponto decimal e sem separador de milhar (se PASSAR, escreva 0)\n"
        "MOTIVO: uma frase curta"
    ))

def ler_lance(texto: str) -> dict:
    campos = {}
    for linha in texto.splitlines():
        if ":" in linha:
            chave, valor = linha.split(":", 1)
            campos[re.sub(r"[^A-Z]", "", chave.upper())] = valor.strip()
    return {
        "acao": re.sub(r"[^A-Z]", "", campos.get("ACAO", "").upper()),
        "lance": ler_preco(campos.get("LANCE", "")),
        "motivo": campos.get("MOTIVO", ""),
    }

## Passo 13 — A rodada de lances

O Python conduz a disputa: pergunta a cada comprador ativo (menos quem lidera), mostra o histórico de lances e **valida** cada lance.
Lance abaixo do mínimo ou acima do próprio orçamento é rejeitado e o comprador sai da disputa, com o motivo registrado.

In [62]:
def rodada_de_lances(id_produto: str, preco_inicial: float, escala: float = 1.0) -> dict:
    agentes = {nome: criar_agente_comprador(nome, id_produto, escala) for nome in compradores}
    ativos = list(compradores)
    lances, eventos = [], []
    lider, lance_atual = None, None

    for rodada in range(1, MAX_RODADAS_LANCES + 1):
        houve_lance = False
        for nome in list(ativos):
            if nome == lider:
                continue
            minimo = preco_inicial if lance_atual is None else lance_atual + INCREMENTO_MINIMO
            historico = "\n".join(f"- rodada {l['rodada']}: {l['nome']} deu {l['valor']:.2f}" for l in lances) or "- nenhum lance ainda"
            mensagem = (
                f"Rodada {rodada}.\n"
                f"Lance atual: {f'{lance_atual:.2f}' if lance_atual else 'nenhum (preço inicial ' + f'{preco_inicial:.2f})'}\n"
                f"Líder: {lider or 'ninguém'}\n"
                f"Lance mínimo para cobrir: {minimo:.2f}\n"
                f"Histórico de lances:\n{historico}"
            )
            r = ler_lance(chamar(agentes[nome], mensagem, f"{nome} - rodada {rodada}"))
            orcamento = orcamento_comprador(nome, id_produto, escala)

            if r["acao"] != "COBRIR":
                motivo = f"passou: {r['motivo']}"
            elif r["lance"] is None or r["lance"] < minimo:
                motivo = f"lance rejeitado: {r['lance']} abaixo do mínimo {minimo:.2f}"
            elif r["lance"] > orcamento:
                motivo = f"lance rejeitado: {r['lance']:.2f} acima do próprio orçamento"
            else:
                lances.append({"rodada": rodada, "nome": nome, "valor": r["lance"]})
                lider, lance_atual, houve_lance = nome, r["lance"], True
                continue

            ativos.remove(nome)
            eventos.append({"rodada": rodada, "nome": nome, "motivo": motivo})
            print(f"[Protocolo] {nome} saiu da disputa ({motivo})")

        if not houve_lance:
            break

    return {"lances": lances, "eventos": eventos, "lider": lider, "lance_atual": lance_atual}

## Passo 14 — Alocação e pagamento (decisões separadas)

- **Alocação:** quem vence. Regra: o maior lance válido. Sem lance → `None` (sem venda, sem forçar um vencedor).
- **Pagamento:** quanto o vencedor paga. Duas regras para comparar:
  - `lance_vencedor`: paga o próprio lance (leilão inglês tradicional);
  - `segundo_maior_lance`: paga o maior lance dos outros, nunca abaixo do preço inicial.

In [63]:
def alocar(resultado_lances: dict):
    if resultado_lances["lider"] is None:
        return None
    return {"vencedor": resultado_lances["lider"], "lance_vencedor": resultado_lances["lance_atual"]}

def calcular_pagamento(alocacao, lances: list, preco_inicial: float, regra: str):
    if alocacao is None:
        return None
    if regra == "lance_vencedor":
        return alocacao["lance_vencedor"]
    if regra == "segundo_maior_lance":
        outros = [l["valor"] for l in lances if l["nome"] != alocacao["vencedor"]]
        return max(max(outros, default=preco_inicial), preco_inicial)
    raise ValueError(f"Regra de pagamento desconhecida: {regra}")

## Passo 15 — Ferramentas do leiloeiro

O leiloeiro conduz o leilão, mas não decide nada "de cabeça": cada etapa é uma ferramenta que roda o protocolo em Python.
O estado do leilão fica em `estado_leilao`, assim o modelo não precisa copiar dados nos argumentos.

In [64]:
from langchain_core.tools import tool

estado_leilao = {}

def novo_estado(id_produto, limite_chegada, escala, regra_pagamento):
    return {
        "anuncio": {**criar_anuncio(id_produto, limite_chegada), "regra_pagamento": regra_pagamento},
        "escala": escala,
        "preco_inicial": None, "motivos_precificacao": {},
        "lances": None, "resultado": None,
        "etapas": [],
    }

@tool
def preparar_leilao() -> str:
    """Coleta as avaliações e define o preço inicial do produto anunciado."""
    rodada = coletar_propostas(estado_leilao["anuncio"])
    escolha = fluxo_precificacao_leilao.invoke(rodada)
    estado_leilao["preco_inicial"] = escolha["preco_inicial"] if escolha else None
    estado_leilao["motivos_precificacao"] = motivos_leilao(rodada)
    estado_leilao["etapas"].append("preparar")
    if escolha is None:
        return ("Sem preço inicial válido. Não abra os lances: chame encerrar_leilao.\n"
                f"Motivos: {estado_leilao['motivos_precificacao']}")
    return (f"Preço inicial definido: {escolha['preco_inicial']:.2f} (avaliador {escolha['avaliador']}). "
            "Agora chame abrir_lances.")

@tool
def abrir_lances() -> str:
    """Conduz a disputa de lances entre os compradores a partir do preço inicial."""
    if estado_leilao["preco_inicial"] is None:
        return "Não há preço inicial: chame encerrar_leilao."
    estado_leilao["lances"] = rodada_de_lances(
        estado_leilao["anuncio"]["produto_id"], estado_leilao["preco_inicial"], estado_leilao["escala"])
    estado_leilao["etapas"].append("lances")
    l = estado_leilao["lances"]
    return f"Lances: {l['lances']}\nSaídas: {l['eventos']}\nAgora chame encerrar_leilao."

@tool
def encerrar_leilao() -> str:
    """Aplica a alocação e a regra de pagamento e registra o resultado final (ou None)."""
    l = estado_leilao["lances"] or {"lances": [], "eventos": [], "lider": None, "lance_atual": None}
    alocacao = alocar(l)
    pagamento = calcular_pagamento(alocacao, l["lances"], estado_leilao["preco_inicial"],
                                   estado_leilao["anuncio"]["regra_pagamento"])
    estado_leilao["resultado"] = None if alocacao is None else {
        "produto_id": estado_leilao["anuncio"]["produto_id"],
        **alocacao,
        "regra_pagamento": estado_leilao["anuncio"]["regra_pagamento"],
        "pagamento": pagamento,
    }
    estado_leilao["etapas"].append("encerrar")
    motivo = "" if alocacao else (
        "Sem preço inicial válido." if estado_leilao["preco_inicial"] is None
        else "Nenhum comprador cobriu o preço inicial.")
    return f"Resultado: {estado_leilao['resultado']} {motivo}\nSaídas: {l['eventos']}"

## Passo 16 — O agente leiloeiro e a função `rodar_leilao`

O leiloeiro chama as ferramentas na ordem do protocolo e anuncia o resultado.
Se ele pular alguma etapa, `rodar_leilao` executa a etapa que faltou em Python, para o registro nunca ficar incompleto.

In [65]:
leiloeiro = create_agent(modelo, tools=[preparar_leilao, abrir_lances, encerrar_leilao], system_prompt=(
    'Você é o Leiloeiro. Você não define preços nem vencedores por conta própria: use as ferramentas, '
    'uma por vez, nesta ordem:\n'
    '1. preparar_leilao;\n'
    '2. abrir_lances (somente se houver preço inicial);\n'
    '3. encerrar_leilao.\n'
    'Depois anuncie em português, de forma curta: o preço inicial, o vencedor e quanto ele paga, '
    'OU "Sem venda" com o motivo. Se o resultado for None, NUNCA indique um vencedor.'
))

def rodar_leilao(id_produto: str, limite_chegada: int = 2, escala_orcamento: float = 1.0,
                 regra_pagamento: str = "lance_vencedor", titulo: str = ""):
    print(f"\n{'=' * 10} {titulo or id_produto} {'=' * 10}")
    estado_leilao.clear()
    estado_leilao.update(novo_estado(id_produto, limite_chegada, escala_orcamento, regra_pagamento))
    inicio = time.perf_counter()

    chamar(leiloeiro, f"Conduza o leilão do produto {id_produto}: {produtos_leilao[id_produto]['nome']}.", "Leiloeiro")

    # Protocolo garantido: executa em Python qualquer etapa que o leiloeiro tenha pulado
    if "preparar" not in estado_leilao["etapas"]:
        print("[Aviso] Leiloeiro pulou preparar_leilao; executando em Python.")
        preparar_leilao.invoke({})
    if estado_leilao["preco_inicial"] is not None and "lances" not in estado_leilao["etapas"]:
        print("[Aviso] Leiloeiro pulou abrir_lances; executando em Python.")
        abrir_lances.invoke({})
    if "encerrar" not in estado_leilao["etapas"]:
        print("[Aviso] Leiloeiro pulou encerrar_leilao; executando em Python.")
        encerrar_leilao.invoke({})

    print(f"\nPreço inicial: {estado_leilao['preco_inicial']}")
    print(f"Resultado registrado: {estado_leilao['resultado']}")
    print(f"Tempo total: {time.perf_counter() - inicio:.1f}s")
    return estado_leilao["resultado"]

## Passo 17 — Caso-base: product01

Orçamentos: Rui 2850, Lia 2100, Davi 1800. Esperado: Davi não consegue cobrir o preço inicial; Rui e Lia disputam e o vencedor paga o próprio lance.

In [66]:
resultado_base = rodar_leilao("product01", titulo="Caso-base: product01, paga o próprio lance")


========== Caso-base: product01, paga o próprio lance ==========

[Avaliador_Senior] (9.1s):
DECISAO: PROPOR
PRECO: 1500.00
JUSTIFICATIVA: O item é uma antiguidade de alta relevância com 85 anos, então estabeleço o preço inicial no custo para atrair muitos lances.

[Agente_Preco_Justo] (13.8s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: O alto valor histórico de um relógio de bolso de 85 anos justifica um preço acima do custo, equilibrando atração e valorização.

[Avaliador_Expert] (10.8s):
DECISAO: PROPOR
PRECO: 1850.00
JUSTIFICATIVA: O item possui 85 anos de antiguidade e alta relevância no mercado, justificando um valor inicial superior ao custo de aquisição.

[Barone_Leiloes] (12.3s):
DECISAO: PROPOR
PRECO: 2150.00
JUSTIFICATIVA: O relógio de bolso vintage possui alta relevância no mercado devido aos seus 85 anos de história, permitindo um preço acima do custo mas acessível para atrair lances.

[Colecionador_Rui - rodada 1] (8.9s):
ACAO: COBRIR
LANCE: 1850.00
MOTIVO: Preço inic

## Passo 18 — Mudando só a regra de pagamento

Mesmo produto e mesmos compradores; só a **regra de pagamento** muda. A alocação (quem vence) segue a mesma lógica, mas o valor pago passa a ser o segundo maior lance.

In [67]:
resultado_segundo = rodar_leilao("product01", regra_pagamento="segundo_maior_lance",
                                 titulo="product01, paga o segundo maior lance")


========== product01, paga o segundo maior lance ==========

[Avaliador_Senior] (7.5s):
DECISAO: PROPOR
PRECO: 1500.00
JUSTIFICATIVA: O item é uma antiguidade de alta relevância com 85 anos, então estabeleço um preço inicial baixo próximo ao custo para atrair muitos lances.

[Agente_Preco_Justo] (7.5s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: O alto valor histórico de um relógio de bolso de 85 anos justifica um preço acima do custo, equilibrando atratividade e valorização.

[Avaliador_Expert] (14.1s):
DECISAO: PROPOR
PRECO: 1850.00
JUSTIFICATIVA: O relógio de bolso possui 85 anos de idade, alta relevância no mercado e pertence à minha especialidade em antiguidades.

[Barone_Leiloes] (16.3s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: A alta relevância de mercado e os 85 anos de idade do relógio justificam um valor acima do custo, equilibrando atração de lances com valorização da antiguidade.

[Colecionador_Rui - rodada 1] (11.0s):
ACAO: COBRIR
LANCE: 1850.00
MOTIVO: Preço ini

## Passo 19 — Leilão sem venda

Orçamentos reduzidos pela metade (`escala_orcamento=0.5`): nenhum comprador alcança o preço inicial.
Esperado: resultado `None`, com o motivo de cada saída. Sem venda é um resultado válido; reabrir seria uma **nova** rodada com condições explícitas.

In [68]:
resultado_sem_venda = rodar_leilao("product01", escala_orcamento=0.5, titulo="Sem venda: orçamentos pela metade")


========== Sem venda: orçamentos pela metade ==========

[Avaliador_Senior] (12.2s):
DECISAO: PROPOR
PRECO: 1500.00
JUSTIFICATIVA: O item é uma antiguidade de alta relevância com 85 anos, então estabeleço um preço inicial baixo próximo ao custo para atrair muitos lances.

[Agente_Preco_Justo] (13.4s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: O alto valor histórico de um relógio de bolso de 85 anos justifica um preço acima do custo, equilibrando atratividade e valorização.

[Avaliador_Expert] (12.6s):
DECISAO: PROPOR
PRECO: 4500.00
JUSTIFICATIVA: O relógio de bolso possui 85 anos de idade, alta relevância no mercado e pertence à categoria de antiguidades que domino perfeitamente.

[Barone_Leiloes] (14.9s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: A alta relevância de mercado e os 85 anos de idade do relógio justificam um valor acima do custo, equilibrando atração de lances e valorização da antiguidade.

[Colecionador_Rui - rodada 1] (9.4s):
ACAO: PASSAR
LANCE: 0
MOTIVO: O pre

## Passo 20 — Leiloar todos os produtos

O mesmo protocolo aplicado aos quatro produtos. Ao final, um resumo com o preço inicial, o vencedor e o pagamento de cada um.

In [69]:
resumo = {}
for id_produto, produto in produtos_leilao.items():
    resultado = rodar_leilao(id_produto, titulo=f"{id_produto}: {produto['nome']}")
    resumo[id_produto] = {
        "produto": produto["nome"],
        "preco_inicial": estado_leilao["preco_inicial"],
        "vencedor": resultado["vencedor"] if resultado else None,
        "pagamento": resultado["pagamento"] if resultado else None,
    }

print("\n=== RESUMO DO LEILÃO ===")
for id_produto, r in resumo.items():
    print(f"- {r['produto']}: preço inicial {r['preco_inicial']}, vencedor {r['vencedor']}, pagamento {r['pagamento']}")


========== product01: Relógio de Bolso Vintage ==========

[Avaliador_Senior] (14.3s):
DECISAO: PROPOR
PRECO: 1500.00
JUSTIFICATIVA: O item é uma antiguidade de alta relevância com 85 anos, então estabeleço um preço inicial baixo próximo ao custo para atrair muitos lances.

[Agente_Preco_Justo] (9.1s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: O alto valor histórico de um relógio de bolso de 85 anos justifica um preço acima do custo, equilibrando atratividade e valorização.

[Avaliador_Expert] (12.2s):
DECISAO: PROPOR
PRECO: 1850.00
JUSTIFICATIVA: O relógio de bolso possui 85 anos de idade, alta relevância no mercado e pertence à minha especialidade em antiguidades.

[Barone_Leiloes] (13.1s):
DECISAO: PROPOR
PRECO: 2850.00
JUSTIFICATIVA: A alta relevância de mercado e os 85 anos de idade do relógio justificam um valor acima do custo, equilibrando atração de lances e valorização da antiguidade.

[Colecionador_Rui - rodada 1] (9.9s):
ACAO: COBRIR
LANCE: 1850.00
MOTIVO: Preço inicial